In [ ]:
# ================================
# 1. Imports
# ================================
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# ================================
# 2. Load environment variables
# ================================
load_dotenv()

API_KEY = os.getenv("OPENROUTER_API_KEY")

if API_KEY is None:
    raise ValueError("OPENROUTER_API_KEY not found in .env")

# ================================
# 3. Initialize LLM
# ================================
llm = ChatOpenAI(
    model="openai/gpt-4.1-mini",   # можно менять модель
    base_url="https://openrouter.ai/api/v1",
    api_key=API_KEY,
    temperature=0,                 # для стабильных решений агента
    max_tokens=2000
)

# ================================
# 4. (Optional) Test call
# ================================
# response = llm.invoke("Say hello")
# print(response.content)

In [ ]:

"""
Universal LangGraph-based Data Description Agent.

This agent is dataset-agnostic.

Responsibilities:
- Load train/test/sample_submission datasets.
- Infer target column when possible, or use a manually provided target.
- Infer task type when possible.
- Describe dataset structure.
- Detect feature groups:
  numeric, categorical, text, datetime, boolean_like, id, target.
- Detect data quality issues:
  missing values, duplicates, outliers, constant columns, high-cardinality columns,
  train/test schema mismatches, class imbalance for classification.
- Ask LLM to review the schema and summarize risks.
- Save profiling artifacts.
- Return a unified response for the orchestrator.

Important:
- This agent does NOT prepare or transform the dataset.
- It only describes the data and passes structured summary to the orchestrator.
- Actual preprocessing should be done manually or by a separate Preparation Agent.

Main public function:
    run_data_description_agent(...)

Example:
    result = run_data_description_agent(
        llm=llm,
        train_path="data/train.csv",
        test_path="data/test.csv",
        sample_submission_path="data/sample_submission.csv",
        target_column=None,
        task_type=None,
        business_task="Predict target value for new rows",
        run_id="demo_run_001",
    )
"""

from __future__ import annotations

import json
import re
import uuid
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, TypedDict

import numpy as np
import pandas as pd

try:
    from langgraph.graph import StateGraph, START, END
except ImportError as exc:
    raise ImportError(
        "LangGraph is not installed. Install it with: pip install langgraph"
    ) from exc


# ============================================================
# 1. State schema
# ============================================================

class DataDescriptionState(TypedDict, total=False):
    # Input
    run_id: str
    train_path: str
    test_path: Optional[str]
    sample_submission_path: Optional[str]
    target_column: Optional[str]
    task_type: Optional[str]
    business_task: str
    artifacts_dir: str

    # Metadata
    train_shape: List[int]
    test_shape: Optional[List[int]]
    sample_submission_columns: List[str]
    train_columns: List[str]
    test_columns: List[str]
    columns_only_in_train: List[str]
    columns_only_in_test: List[str]

    # Dataset summaries
    schema: Dict[str, List[str]]
    basic_profile: Dict[str, Any]
    data_quality: Dict[str, Any]
    target_properties: Dict[str, Any]

    # LLM outputs
    llm_schema_review: Dict[str, Any]
    llm_data_risk_summary: Dict[str, Any]

    # Artifacts
    artifacts: Dict[str, str]

    # Logs and status
    logs: List[Dict[str, Any]]
    status: str
    reason: Optional[str]


# ============================================================
# 2. Utility functions
# ============================================================

def now_iso() -> str:
    return datetime.utcnow().isoformat(timespec="seconds") + "Z"


def make_log(agent: str, message: str, level: str = "INFO") -> Dict[str, Any]:
    return {
        "ts": now_iso(),
        "agent": agent,
        "level": level,
        "message": message,
    }


def ensure_dir(path: str | Path) -> Path:
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def safe_json_dump(payload: Dict[str, Any], path: str | Path) -> None:
    path = Path(path)
    ensure_dir(path.parent)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=4, ensure_ascii=False, default=str)


def safe_markdown_dump(text: str, path: str | Path) -> None:
    path = Path(path)
    ensure_dir(path.parent)
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)


def extract_text_from_llm_response(response: Any) -> str:
    if response is None:
        return ""
    if isinstance(response, str):
        return response
    if hasattr(response, "content"):
        return str(response.content)
    return str(response)


def call_llm(llm: Any, prompt: str) -> str:
    response = llm.invoke(prompt)
    return extract_text_from_llm_response(response)


def extract_json_from_text(text: str) -> Dict[str, Any]:
    text = text.strip()

    fenced = re.search(r"```(?:json)?\s*(.*?)```", text, flags=re.DOTALL)
    if fenced:
        text = fenced.group(1).strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    start = text.find("{")
    end = text.rfind("}")

    if start != -1 and end != -1 and end > start:
        return json.loads(text[start:end + 1])

    raise ValueError(f"Could not parse JSON from LLM output:\n{text[:1000]}")


def to_serializable(value: Any) -> Any:
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        if np.isnan(value) or np.isinf(value):
            return None
        return float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if pd.isna(value):
        return None
    return value


def get_run_dir(state: DataDescriptionState) -> Path:
    return ensure_dir(Path(state["artifacts_dir"]) / state["run_id"] / "data_description")


# ============================================================
# 3. Universal inference helpers
# ============================================================

def infer_target_column(
    train_columns: List[str],
    test_columns: Optional[List[str]] = None,
    sample_submission_columns: Optional[List[str]] = None,
    provided_target: Optional[str] = None,
) -> Optional[str]:
    """
    Universal target inference.

    Priority:
    1. Manually provided target.
    2. If train/test are provided and exactly one column exists only in train.
    3. If sample_submission has one non-id column that exists in train.
    4. Common target-like names.
    5. None.
    """
    if provided_target:
        return provided_target

    if test_columns is not None:
        only_in_train = [col for col in train_columns if col not in test_columns]
        if len(only_in_train) == 1:
            return only_in_train[0]

    if sample_submission_columns:
        id_like = {"id", "ID", "Id", "index", "row_id", "RowId"}
        candidates = [
            col for col in sample_submission_columns
            if col in train_columns and col not in id_like
        ]
        if len(candidates) == 1:
            return candidates[0]

    common_target_names = [
        "target",
        "label",
        "y",
        "class",
        "outcome",
        "response",
        "price",
        "SalePrice",
        "churn",
        "survived",
        "income",
    ]

    for name in common_target_names:
        if name in train_columns:
            return name

    return None


def infer_task_type(y: Optional[pd.Series], provided_task_type: Optional[str] = None) -> str:
    """
    Universal task type inference.
    Returns classification, regression, or unknown.
    """
    if provided_task_type in {"classification", "regression"}:
        return provided_task_type

    if y is None:
        return "unknown"

    non_null = y.dropna()

    if len(non_null) == 0:
        return "unknown"

    unique_count = int(non_null.nunique(dropna=True))
    unique_ratio = unique_count / max(len(non_null), 1)

    if pd.api.types.is_numeric_dtype(non_null):
        # Numeric target with many values is usually regression.
        if unique_count > 20 and unique_ratio > 0.02:
            return "regression"
        return "classification"

    return "classification"


def detect_id_columns(df: pd.DataFrame, columns: List[str], target_column: Optional[str]) -> List[str]:
    id_cols = []

    for col in columns:
        if col == target_column:
            continue

        lowered = col.lower()
        unique_ratio = df[col].nunique(dropna=True) / max(len(df), 1)

        if lowered in {"id", "row_id", "index"} or lowered.endswith("_id") or lowered.endswith("id"):
            id_cols.append(col)
            continue

        # A nearly unique integer/object column is probably an identifier.
        if unique_ratio >= 0.98 and (
            pd.api.types.is_integer_dtype(df[col]) or "id" in lowered
        ):
            id_cols.append(col)

    return sorted(set(id_cols))


def detect_datetime_columns(df: pd.DataFrame, columns: List[str], sample_size: int = 1000) -> List[str]:
    datetime_cols = []

    name_hints = [
        "date", "time", "timestamp", "created", "updated",
        "year", "month", "day", "since"
    ]

    for col in columns:
        series = df[col].dropna()

        if len(series) == 0:
            continue

        name_looks_datetime = any(hint in col.lower() for hint in name_hints)

        # Skip arbitrary numeric columns unless their name looks date-like.
        if pd.api.types.is_numeric_dtype(series) and not name_looks_datetime:
            continue

        sample = series.astype(str).head(sample_size)

        parsed = pd.to_datetime(sample, errors="coerce", utc=False)
        valid_ratio = parsed.notna().mean()

        if valid_ratio >= 0.70:
            datetime_cols.append(col)

    return sorted(set(datetime_cols))


def detect_boolean_like_columns(df: pd.DataFrame, columns: List[str]) -> List[str]:
    boolean_cols = []

    allowed = {
        "true", "false", "t", "f", "yes", "no", "y", "n",
        "0", "1", "0.0", "1.0"
    }

    for col in columns:
        values = set(df[col].dropna().astype(str).str.strip().str.lower().unique())

        if len(values) == 0:
            continue

        if len(values) <= 2 and values.issubset(allowed):
            boolean_cols.append(col)

    return sorted(set(boolean_cols))


def detect_text_columns(df: pd.DataFrame, object_cols: List[str]) -> List[str]:
    text_cols = []

    name_hints = [
        "text", "description", "desc", "comment", "review", "summary",
        "message", "content", "body", "about", "notes", "title", "name"
    ]

    for col in object_cols:
        series = df[col].dropna().astype(str)

        if len(series) == 0:
            continue

        avg_len = float(series.str.len().mean())
        max_len = int(series.str.len().max())
        unique_ratio = series.nunique(dropna=True) / max(len(series), 1)

        name_looks_text = any(hint in col.lower() for hint in name_hints)

        # Text is usually longer and more unique than regular categorical columns.
        if name_looks_text and avg_len >= 20:
            text_cols.append(col)
        elif avg_len >= 50 and max_len >= 100 and unique_ratio > 0.20:
            text_cols.append(col)

    return sorted(set(text_cols))


def infer_schema(df: pd.DataFrame, target_column: Optional[str]) -> Dict[str, List[str]]:
    all_columns = list(df.columns)

    id_cols = detect_id_columns(df, all_columns, target_column)

    candidate_columns = [
        col for col in all_columns
        if col not in id_cols and col != target_column
    ]

    datetime_cols = detect_datetime_columns(df, candidate_columns)

    remaining = [
        col for col in candidate_columns
        if col not in datetime_cols
    ]

    numeric_cols = [
        col for col in remaining
        if pd.api.types.is_numeric_dtype(df[col])
    ]

    object_cols = [
        col for col in remaining
        if pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_string_dtype(df[col])
    ]

    boolean_like_cols = detect_boolean_like_columns(df, remaining)

    text_cols = detect_text_columns(df, object_cols)

    categorical_cols = [
        col for col in remaining
        if col not in numeric_cols
        and col not in text_cols
        and col not in datetime_cols
        and col != target_column
    ]

    # Boolean-like columns are also usually categorical, but tracked separately.
    for col in boolean_like_cols:
        if col not in categorical_cols and col not in numeric_cols:
            categorical_cols.append(col)

    return {
        "numeric": sorted(set(numeric_cols) - set(boolean_like_cols)),
        "categorical": sorted(set(categorical_cols)),
        "text": sorted(set(text_cols)),
        "datetime": sorted(set(datetime_cols)),
        "boolean_like": sorted(set(boolean_like_cols)),
        "id": sorted(set(id_cols)),
        "target": [target_column] if target_column else [],
    }


# ============================================================
# 4. Profiling helpers
# ============================================================

def summarize_missing(df: pd.DataFrame, top_n: int = 50) -> Dict[str, Any]:
    missing_count = df.isna().sum()
    missing_pct = (missing_count / max(len(df), 1)).round(4)

    rows = []

    for col in df.columns:
        count = int(missing_count[col])
        if count > 0:
            rows.append({
                "column": col,
                "missing_count": count,
                "missing_pct": float(missing_pct[col]),
            })

    rows = sorted(rows, key=lambda x: x["missing_pct"], reverse=True)

    return {
        "total_missing_cells": int(missing_count.sum()),
        "columns_with_missing": len(rows),
        "top_missing_columns": rows[:top_n],
    }


def summarize_numeric(df: pd.DataFrame, numeric_cols: List[str], top_n: int = 80) -> Dict[str, Any]:
    result = {}

    for col in numeric_cols[:top_n]:
        series = df[col].dropna()

        if len(series) == 0:
            result[col] = {
                "count": 0,
                "mean": None,
                "std": None,
                "min": None,
                "q25": None,
                "median": None,
                "q75": None,
                "max": None,
                "zero_count": None,
            }
            continue

        result[col] = {
            "count": int(series.count()),
            "mean": to_serializable(series.mean()),
            "std": to_serializable(series.std()),
            "min": to_serializable(series.min()),
            "q25": to_serializable(series.quantile(0.25)),
            "median": to_serializable(series.median()),
            "q75": to_serializable(series.quantile(0.75)),
            "max": to_serializable(series.max()),
            "zero_count": int((series == 0).sum()),
        }

    return result


def summarize_cardinality(df: pd.DataFrame, columns: List[str], top_n: int = 80) -> Dict[str, Any]:
    rows = []

    for col in columns:
        nunique = int(df[col].nunique(dropna=True))
        unique_ratio = round(nunique / max(len(df), 1), 4)

        sample_values = [
            str(x)[:120]
            for x in df[col].dropna().unique()[:5]
        ]

        rows.append({
            "column": col,
            "unique_values": nunique,
            "unique_ratio": unique_ratio,
            "sample_values": sample_values,
        })

    rows = sorted(rows, key=lambda x: x["unique_values"], reverse=True)

    high_cardinality = [
        row for row in rows
        if row["unique_values"] > 50 or row["unique_ratio"] > 0.20
    ]

    return {
        "high_cardinality_columns": high_cardinality[:top_n],
        "all_cardinality": rows[:top_n],
    }


def detect_outliers_iqr(df: pd.DataFrame, numeric_cols: List[str], top_n: int = 50) -> Dict[str, Any]:
    rows = []

    for col in numeric_cols:
        series = df[col].dropna()

        if len(series) < 20:
            continue

        q1 = series.quantile(0.25)
        q3 = series.quantile(0.75)
        iqr = q3 - q1

        if pd.isna(iqr) or iqr == 0:
            continue

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        count = int(((series < lower) | (series > upper)).sum())
        pct = round(count / max(len(series), 1), 4)

        if count > 0:
            rows.append({
                "column": col,
                "outlier_count": count,
                "outlier_pct": pct,
                "lower_bound": to_serializable(lower),
                "upper_bound": to_serializable(upper),
                "min": to_serializable(series.min()),
                "max": to_serializable(series.max()),
            })

    rows = sorted(rows, key=lambda x: x["outlier_pct"], reverse=True)

    return {
        "method": "IQR",
        "outlier_columns": rows[:top_n],
    }


def summarize_target(y: Optional[pd.Series], task_type: str) -> Dict[str, Any]:
    if y is None:
        return {
            "target_column": None,
            "task_type": task_type,
            "available": False,
        }

    non_null = y.dropna()

    base = {
        "target_column": y.name,
        "task_type": task_type,
        "available": True,
        "missing_count": int(y.isna().sum()),
        "missing_pct": round(float(y.isna().mean()), 4),
        "unique_values": int(non_null.nunique(dropna=True)),
    }

    if len(non_null) == 0:
        return base

    if task_type == "regression" and pd.api.types.is_numeric_dtype(non_null):
        q1 = non_null.quantile(0.25)
        q3 = non_null.quantile(0.75)
        iqr = q3 - q1

        if pd.notna(iqr) and iqr != 0:
            lower = q1 - 1.5 * iqr
            upper = q3 + 1.5 * iqr
            outlier_count = int(((non_null < lower) | (non_null > upper)).sum())
        else:
            outlier_count = 0

        skew = non_null.skew()

        base.update({
            "mean": to_serializable(non_null.mean()),
            "std": to_serializable(non_null.std()),
            "min": to_serializable(non_null.min()),
            "q25": to_serializable(q1),
            "median": to_serializable(non_null.median()),
            "q75": to_serializable(q3),
            "max": to_serializable(non_null.max()),
            "skew": to_serializable(skew),
            "outlier_count_iqr": outlier_count,
            "is_positive_numeric": bool((non_null > 0).all()),
            "is_highly_skewed": bool(abs(skew) > 1.0) if pd.notna(skew) else False,
        })

    elif task_type == "classification":
        counts = non_null.value_counts(dropna=False)

        distribution = {
            str(k): {
                "count": int(v),
                "pct": round(float(v / len(non_null)), 4),
            }
            for k, v in counts.items()
        }

        majority_pct = float(counts.iloc[0] / len(non_null)) if len(counts) else None

        base.update({
            "class_distribution": distribution,
            "majority_class_pct": round(majority_pct, 4) if majority_pct is not None else None,
            "is_imbalanced": bool(majority_pct is not None and majority_pct >= 0.70),
        })

    return base


def compare_train_test_schema(
    train_df: pd.DataFrame,
    test_df: Optional[pd.DataFrame],
    target_column: Optional[str],
) -> Dict[str, Any]:
    if test_df is None:
        return {
            "test_provided": False,
            "common_feature_columns": None,
            "missing_in_test": [],
            "extra_in_test": [],
            "dtype_mismatches": [],
        }

    train_cols = set(train_df.columns)
    test_cols = set(test_df.columns)

    feature_train_cols = train_cols - ({target_column} if target_column else set())

    common_cols = sorted(feature_train_cols & test_cols)
    missing_in_test = sorted(feature_train_cols - test_cols)
    extra_in_test = sorted(test_cols - feature_train_cols)

    dtype_mismatches = []

    for col in common_cols:
        train_dtype = str(train_df[col].dtype)
        test_dtype = str(test_df[col].dtype)

        if train_dtype != test_dtype:
            dtype_mismatches.append({
                "column": col,
                "train_dtype": train_dtype,
                "test_dtype": test_dtype,
            })

    return {
        "test_provided": True,
        "common_feature_columns": len(common_cols),
        "missing_in_test": missing_in_test,
        "extra_in_test": extra_in_test,
        "dtype_mismatches": dtype_mismatches,
    }


# ============================================================
# 5. Prompts
# ============================================================

def build_schema_review_prompt(state: DataDescriptionState) -> str:
    compact_profile = {
        "business_task": state.get("business_task", ""),
        "train_shape": state.get("train_shape"),
        "test_shape": state.get("test_shape"),
        "target_column": state.get("target_column"),
        "task_type": state.get("task_type"),
        "schema_detected_by_code": state.get("schema"),
        "columns_only_in_train": state.get("columns_only_in_train"),
        "columns_only_in_test": state.get("columns_only_in_test"),
        "sample_submission_columns": state.get("sample_submission_columns"),
        "top_missing_columns": state.get("basic_profile", {}).get("missing", {}).get("top_missing_columns", [])[:20],
        "high_cardinality_columns": state.get("basic_profile", {}).get("cardinality", {}).get("high_cardinality_columns", [])[:20],
        "target_properties": state.get("target_properties", {}),
    }

    return f"""
You are a universal Data Description Agent inside a multi-agent ML workflow.

Your task:
1. Review the automatically detected feature schema.
2. Correct obvious mistakes if needed.
3. Identify possible leakage-risk columns using only column names and dataset context.
4. Identify columns that are likely IDs or should not be used directly as model features.
5. Return only valid JSON.

Dataset profile:
{json.dumps(compact_profile, indent=2, ensure_ascii=False)}

Return JSON with exactly this schema:
{{
  "schema_review_status": "accepted_or_corrected",
  "schema": {{
    "numeric": [],
    "categorical": [],
    "text": [],
    "datetime": [],
    "boolean_like": [],
    "id": [],
    "target": []
  }},
  "possible_leakage_risk_columns": [],
  "possible_drop_candidates": [],
  "notes_for_orchestrator": [],
  "reason": "short explanation"
}}

Rules:
- Do not invent columns.
- Keep target only in the target group.
- Do not perform preprocessing.
- Do not recommend dataset-specific transformations here.
- Return JSON only.
""".strip()


def build_data_risk_summary_prompt(state: DataDescriptionState) -> str:
    compact = {
        "business_task": state.get("business_task", ""),
        "target_column": state.get("target_column"),
        "task_type": state.get("task_type"),
        "train_shape": state.get("train_shape"),
        "test_shape": state.get("test_shape"),
        "schema": state.get("schema"),
        "data_quality": state.get("data_quality"),
        "target_properties": state.get("target_properties"),
        "llm_schema_review": state.get("llm_schema_review"),
    }

    return f"""
You are a Data Risk Summary Agent.

Your task:
Summarize what the orchestrator and the human team should know before preprocessing.
This agent does NOT prepare data and does NOT train models.

Dataset profile:
{json.dumps(compact, indent=2, ensure_ascii=False)}

Return only valid JSON with this schema:
{{
  "business_interpretation": "short interpretation of the ML task",
  "main_data_risks": [],
  "detected_anomalies": [],
  "what_to_check_manually": [],
  "handoff_to_preparation": {{
    "has_numeric_features": true,
    "has_categorical_features": true,
    "has_text_features": true,
    "has_datetime_features": true,
    "has_missing_values": true,
    "has_outliers": true,
    "has_high_cardinality_features": true,
    "has_possible_leakage_risk": true,
    "has_class_imbalance": false
  }},
  "suggested_next_agent": "tabular_preparation_or_text_preparation_or_modeling_or_manual_review",
  "summary_for_orchestrator": "short summary"
}}

Rules:
- Stay dataset-agnostic.
- Do not tell that data was transformed.
- Do not choose final preprocessing methods; only highlight issues for the next step.
- If task_type is unknown, explicitly mention that manual target/task confirmation is needed.
- Return JSON only.
""".strip()


# ============================================================
# 6. Graph nodes
# ============================================================

def load_data_node(state: DataDescriptionState) -> Dict[str, Any]:
    try:
        train_path = Path(state["train_path"])
        test_path = Path(state["test_path"]) if state.get("test_path") else None
        sample_path = Path(state["sample_submission_path"]) if state.get("sample_submission_path") else None

        if not train_path.exists():
            raise FileNotFoundError(f"Train file not found: {train_path}")

        train_df = pd.read_csv(train_path)

        test_df = None
        test_columns: List[str] = []

        if test_path:
            if not test_path.exists():
                raise FileNotFoundError(f"Test file not found: {test_path}")
            test_df = pd.read_csv(test_path)
            test_columns = list(test_df.columns)

        sample_submission_columns: List[str] = []

        if sample_path and sample_path.exists():
            sample_submission_columns = list(pd.read_csv(sample_path, nrows=5).columns)

        train_columns = list(train_df.columns)

        target_column = infer_target_column(
            train_columns=train_columns,
            test_columns=test_columns if test_columns else None,
            sample_submission_columns=sample_submission_columns,
            provided_target=state.get("target_column"),
        )

        if target_column and target_column not in train_df.columns:
            raise ValueError(f"Target column '{target_column}' is not present in train dataset.")

        y = train_df[target_column] if target_column else None

        task_type = infer_task_type(
            y,
            provided_task_type=state.get("task_type"),
        )

        return {
            "train_shape": [int(train_df.shape[0]), int(train_df.shape[1])],
            "test_shape": [int(test_df.shape[0]), int(test_df.shape[1])] if test_df is not None else None,
            "sample_submission_columns": sample_submission_columns,
            "train_columns": train_columns,
            "test_columns": test_columns,
            "columns_only_in_train": sorted(list(set(train_columns) - set(test_columns))) if test_columns else [],
            "columns_only_in_test": sorted(list(set(test_columns) - set(train_columns))) if test_columns else [],
            "target_column": target_column,
            "task_type": task_type,
            "status": "data_loaded",
            "logs": state.get("logs", []) + [
                make_log(
                    "load_data",
                    f"Loaded train={train_df.shape}; "
                    f"test={test_df.shape if test_df is not None else None}; "
                    f"target={target_column}; task_type={task_type}.",
                )
            ],
        }

    except Exception as exc:
        return {
            "status": "error",
            "reason": str(exc),
            "logs": state.get("logs", []) + [
                make_log("load_data", f"Failed to load data: {str(exc)}", level="ERROR")
            ],
        }


def basic_profile_node(state: DataDescriptionState) -> Dict[str, Any]:
    if state.get("status") == "error":
        return {}

    try:
        train_df = pd.read_csv(state["train_path"])

        test_df = None
        if state.get("test_path"):
            test_df = pd.read_csv(state["test_path"])

        target_column = state.get("target_column")

        schema = infer_schema(train_df, target_column)

        y = train_df[target_column] if target_column else None
        target_properties = summarize_target(y, state.get("task_type", "unknown"))

        profile_feature_cols = [
            col for group in ["categorical", "text", "boolean_like", "id"]
            for col in schema.get(group, [])
        ]

        numeric_for_outliers = schema.get("numeric", []).copy()

        if (
            target_column
            and state.get("task_type") == "regression"
            and pd.api.types.is_numeric_dtype(train_df[target_column])
        ):
            numeric_for_outliers.append(target_column)

        basic_profile = {
            "n_rows_train": int(train_df.shape[0]),
            "n_columns_train": int(train_df.shape[1]),
            "n_rows_test": int(test_df.shape[0]) if test_df is not None else None,
            "n_columns_test": int(test_df.shape[1]) if test_df is not None else None,
            "dtypes": {col: str(dtype) for col, dtype in train_df.dtypes.items()},
            "missing": summarize_missing(train_df),
            "duplicates": {
                "train_duplicate_rows": int(train_df.duplicated().sum()),
                "test_duplicate_rows": int(test_df.duplicated().sum()) if test_df is not None else None,
            },
            "numeric_summary": summarize_numeric(train_df, schema.get("numeric", [])),
            "cardinality": summarize_cardinality(train_df, profile_feature_cols),
            "train_test_schema": compare_train_test_schema(train_df, test_df, target_column),
            "outliers": detect_outliers_iqr(train_df, numeric_for_outliers),
        }

        return {
            "schema": schema,
            "basic_profile": basic_profile,
            "target_properties": target_properties,
            "status": "profile_created",
            "logs": state.get("logs", []) + [
                make_log(
                    "basic_profile",
                    "Universal profile, schema and target properties created.",
                )
            ],
        }

    except Exception as exc:
        return {
            "status": "error",
            "reason": str(exc),
            "logs": state.get("logs", []) + [
                make_log("basic_profile", f"Failed to create profile: {str(exc)}", level="ERROR")
            ],
        }


def llm_schema_review_node_factory(llm: Any):
    def llm_schema_review_node(state: DataDescriptionState) -> Dict[str, Any]:
        if state.get("status") == "error":
            return {}

        try:
            prompt = build_schema_review_prompt(state)
            raw = call_llm(llm, prompt)
            review = extract_json_from_text(raw)

            valid_cols = set(state.get("train_columns", []))
            cleaned_schema: Dict[str, List[str]] = {}

            for group in ["numeric", "categorical", "text", "datetime", "boolean_like", "id", "target"]:
                cols = review.get("schema", {}).get(group, [])
                cleaned_schema[group] = [col for col in cols if col in valid_cols]

            target = state.get("target_column")

            if target:
                cleaned_schema["target"] = [target]

                for group in ["numeric", "categorical", "text", "datetime", "boolean_like", "id"]:
                    cleaned_schema[group] = [
                        col for col in cleaned_schema.get(group, [])
                        if col != target
                    ]

            review["schema"] = cleaned_schema

            return {
                "schema": cleaned_schema,
                "llm_schema_review": review,
                "status": "schema_reviewed",
                "logs": state.get("logs", []) + [
                    make_log("llm_schema_review", "LLM reviewed schema.")
                ],
            }

        except Exception as exc:
            fallback = {
                "schema_review_status": "fallback_code_schema_used",
                "schema": state.get("schema", {}),
                "possible_leakage_risk_columns": [],
                "possible_drop_candidates": [],
                "notes_for_orchestrator": [
                    "LLM schema review failed; code-detected schema was used."
                ],
                "reason": str(exc),
            }

            return {
                "llm_schema_review": fallback,
                "status": "schema_review_fallback",
                "logs": state.get("logs", []) + [
                    make_log(
                        "llm_schema_review",
                        f"LLM schema review failed; fallback schema used: {str(exc)}",
                        level="WARNING",
                    )
                ],
            }

    return llm_schema_review_node


def quality_checks_node(state: DataDescriptionState) -> Dict[str, Any]:
    if state.get("status") == "error":
        return {}

    try:
        basic = state.get("basic_profile", {})
        schema = state.get("schema", {})

        missing = basic.get("missing", {})
        duplicates = basic.get("duplicates", {})
        outliers = basic.get("outliers", {})
        cardinality = basic.get("cardinality", {})
        train_test_schema = basic.get("train_test_schema", {})

        constant_columns = []

        train_df = pd.read_csv(state["train_path"])

        for col in train_df.columns:
            if train_df[col].nunique(dropna=False) <= 1:
                constant_columns.append(col)

        mostly_missing_columns = [
            item["column"]
            for item in missing.get("top_missing_columns", [])
            if item.get("missing_pct", 0) >= 0.50
        ]

        possible_leakage_risk_columns = state.get("llm_schema_review", {}).get(
            "possible_leakage_risk_columns",
            [],
        )

        possible_drop_candidates = state.get("llm_schema_review", {}).get(
            "possible_drop_candidates",
            [],
        )

        high_cardinality_columns = [
            row["column"]
            for row in cardinality.get("high_cardinality_columns", [])
        ]

        anomaly_flags = {
            "has_missing_values": bool(missing.get("columns_with_missing", 0) > 0),
            "has_duplicate_rows": bool((duplicates.get("train_duplicate_rows") or 0) > 0),
            "has_outliers": bool(len(outliers.get("outlier_columns", [])) > 0),
            "has_high_cardinality": bool(len(high_cardinality_columns) > 0),
            "has_constant_columns": bool(len(constant_columns) > 0),
            "has_mostly_missing_columns": bool(len(mostly_missing_columns) > 0),
            "has_possible_leakage_risk": bool(len(possible_leakage_risk_columns) > 0),
            "has_train_test_schema_mismatch": bool(
                train_test_schema.get("missing_in_test")
                or train_test_schema.get("extra_in_test")
                or train_test_schema.get("dtype_mismatches")
            ),
            "target_has_missing": bool(state.get("target_properties", {}).get("missing_count", 0) > 0),
            "classification_imbalance": bool(state.get("target_properties", {}).get("is_imbalanced", False)),
            "task_type_unknown": bool(state.get("task_type") == "unknown"),
            "target_unknown": bool(state.get("target_column") is None),
        }

        data_quality = {
            "missing": missing,
            "duplicates": duplicates,
            "outliers": outliers,
            "constant_columns": constant_columns,
            "mostly_missing_columns": mostly_missing_columns,
            "high_cardinality_columns": high_cardinality_columns,
            "possible_leakage_risk_columns": possible_leakage_risk_columns,
            "possible_drop_candidates": possible_drop_candidates,
            "train_test_schema": train_test_schema,
            "target_properties": state.get("target_properties", {}),
            "anomaly_flags": anomaly_flags,
            "feature_group_counts": {
                group: len(cols)
                for group, cols in schema.items()
            },
        }

        return {
            "data_quality": data_quality,
            "status": "quality_checked",
            "logs": state.get("logs", []) + [
                make_log(
                    "quality_checks",
                    "Universal data quality checks completed.",
                )
            ],
        }

    except Exception as exc:
        return {
            "status": "error",
            "reason": str(exc),
            "logs": state.get("logs", []) + [
                make_log("quality_checks", f"Quality checks failed: {str(exc)}", level="ERROR")
            ],
        }


def llm_data_risk_summary_node_factory(llm: Any):
    def llm_data_risk_summary_node(state: DataDescriptionState) -> Dict[str, Any]:
        if state.get("status") == "error":
            return {}

        try:
            prompt = build_data_risk_summary_prompt(state)
            raw = call_llm(llm, prompt)
            risk_summary = extract_json_from_text(raw)

            return {
                "llm_data_risk_summary": risk_summary,
                "status": "risk_summary_created",
                "logs": state.get("logs", []) + [
                    make_log("llm_data_risk_summary", "LLM summarized data risks for orchestrator.")
                ],
            }

        except Exception as exc:
            flags = state.get("data_quality", {}).get("anomaly_flags", {})

            fallback = {
                "business_interpretation": (
                    "LLM risk summary failed. The orchestrator should rely on structured "
                    "profiling outputs and manual review."
                ),
                "main_data_risks": [
                    key for key, value in flags.items()
                    if value is True
                ],
                "detected_anomalies": [],
                "what_to_check_manually": [
                    "Confirm target column and task type.",
                    "Review columns with many missing values.",
                    "Review possible leakage and ID-like columns.",
                ],
                "handoff_to_preparation": {
                    "has_numeric_features": bool(state.get("schema", {}).get("numeric")),
                    "has_categorical_features": bool(state.get("schema", {}).get("categorical")),
                    "has_text_features": bool(state.get("schema", {}).get("text")),
                    "has_datetime_features": bool(state.get("schema", {}).get("datetime")),
                    "has_missing_values": bool(flags.get("has_missing_values")),
                    "has_outliers": bool(flags.get("has_outliers")),
                    "has_high_cardinality_features": bool(flags.get("has_high_cardinality")),
                    "has_possible_leakage_risk": bool(flags.get("has_possible_leakage_risk")),
                    "has_class_imbalance": bool(flags.get("classification_imbalance")),
                },
                "suggested_next_agent": "manual_review",
                "summary_for_orchestrator": "Data description completed with fallback risk summary.",
            }

            return {
                "llm_data_risk_summary": fallback,
                "status": "risk_summary_fallback",
                "logs": state.get("logs", []) + [
                    make_log(
                        "llm_data_risk_summary",
                        f"LLM risk summary failed; fallback used: {str(exc)}",
                        level="WARNING",
                    )
                ],
            }

    return llm_data_risk_summary_node


def packaging_node(state: DataDescriptionState) -> Dict[str, Any]:
    run_dir = get_run_dir(state)

    data_profile_path = run_dir / "data_profile.json"
    schema_path = run_dir / "schema.json"
    quality_report_path = run_dir / "quality_report.json"
    risk_summary_path = run_dir / "risk_summary.json"
    markdown_report_path = run_dir / "data_description_report.md"
    summary_path = run_dir / "data_description_summary.json"

    if state.get("status") == "error":
        summary = {
            "agent": "data_description_agent",
            "status": "error",
            "skipped": False,
            "summary": "Data description failed.",
            "decisions": {},
            "artifacts": {
                "data_description_summary_path": str(summary_path),
            },
            "next_input": {},
            "logs": state.get("logs", []),
            "reason": state.get("reason"),
        }

        safe_json_dump(summary, summary_path)

        return {
            "artifacts": summary["artifacts"],
            "status": "error",
            "reason": state.get("reason"),
            "logs": state.get("logs", []) + [
                make_log("packaging", "Packaged failed Data Description Agent result.", level="ERROR")
            ],
        }

    data_profile = {
        "train_shape": state.get("train_shape"),
        "test_shape": state.get("test_shape"),
        "sample_submission_columns": state.get("sample_submission_columns"),
        "train_columns": state.get("train_columns"),
        "test_columns": state.get("test_columns"),
        "columns_only_in_train": state.get("columns_only_in_train"),
        "columns_only_in_test": state.get("columns_only_in_test"),
        "basic_profile": state.get("basic_profile"),
        "target_properties": state.get("target_properties"),
    }

    safe_json_dump(data_profile, data_profile_path)
    safe_json_dump(state.get("schema", {}), schema_path)
    safe_json_dump(state.get("data_quality", {}), quality_report_path)
    safe_json_dump(state.get("llm_data_risk_summary", {}), risk_summary_path)

    risk_summary = state.get("llm_data_risk_summary", {})

    markdown = f"""# Data Description Agent Report

## Purpose

This report describes the dataset structure and data quality issues.
No preprocessing or transformation was applied by this agent.

## Business task

{state.get("business_task", "")}

## Detected setup

- Target column: `{state.get("target_column")}`
- Task type: `{state.get("task_type")}`
- Train shape: `{state.get("train_shape")}`
- Test shape: `{state.get("test_shape")}`

## Schema

```json
{json.dumps(state.get("schema", {}), indent=2, ensure_ascii=False)}
```

## Data quality flags

```json
{json.dumps(state.get("data_quality", {}).get("anomaly_flags", {}), indent=2, ensure_ascii=False)}
```

## Target properties

```json
{json.dumps(state.get("target_properties", {}), indent=2, ensure_ascii=False)}
```

## Business interpretation

{risk_summary.get("business_interpretation", "")}

## Main data risks

{chr(10).join("- " + str(x) for x in risk_summary.get("main_data_risks", []))}

## What to check manually

{chr(10).join("- " + str(x) for x in risk_summary.get("what_to_check_manually", []))}

## Handoff to preparation

```json
{json.dumps(risk_summary.get("handoff_to_preparation", {}), indent=2, ensure_ascii=False)}
```

## Summary for orchestrator

{risk_summary.get("summary_for_orchestrator", "")}
"""

    safe_markdown_dump(markdown, markdown_report_path)

    artifacts = {
        "data_profile_path": str(data_profile_path),
        "schema_path": str(schema_path),
        "quality_report_path": str(quality_report_path),
        "risk_summary_path": str(risk_summary_path),
        "data_description_report_path": str(markdown_report_path),
        "data_description_summary_path": str(summary_path),
    }

    summary = {
        "agent": "data_description_agent",
        "status": "success",
        "skipped": False,
        "summary": (
            f"Data description completed. "
            f"Target column: {state.get('target_column')}. "
            f"Task type: {state.get('task_type')}. "
            f"Train shape: {state.get('train_shape')}; "
            f"test shape: {state.get('test_shape')}."
        ),
        "decisions": {
            "target_column": state.get("target_column"),
            "task_type": state.get("task_type"),
            "schema": state.get("schema"),
            "data_quality": state.get("data_quality"),
            "target_properties": state.get("target_properties"),
            "llm_schema_review": state.get("llm_schema_review"),
            "llm_data_risk_summary": state.get("llm_data_risk_summary"),
        },
        "artifacts": artifacts,
        "next_input": {
            "target_column": state.get("target_column"),
            "task_type": state.get("task_type"),
            "schema": state.get("schema"),
            "data_summary": {
                "train_shape": state.get("train_shape"),
                "test_shape": state.get("test_shape"),
                "missing": state.get("data_quality", {}).get("missing", {}),
                "duplicates": state.get("data_quality", {}).get("duplicates", {}),
                "outlier_cols": [
                    row["column"]
                    for row in state.get("data_quality", {}).get("outliers", {}).get("outlier_columns", [])
                ],
                "target_properties": state.get("target_properties"),
                "anomaly_flags": state.get("data_quality", {}).get("anomaly_flags", {}),
            },
            "handoff_to_preparation": state.get("llm_data_risk_summary", {}).get("handoff_to_preparation", {}),
            "manual_review_items": state.get("llm_data_risk_summary", {}).get("what_to_check_manually", []),
        },
        "logs": state.get("logs", []),
        "reason": None,
    }

    safe_json_dump(summary, summary_path)

    return {
        "artifacts": artifacts,
        "status": "success",
        "reason": None,
        "logs": state.get("logs", []) + [
            make_log("packaging", "Data description artifacts packaged successfully.")
        ],
    }


# ============================================================
# 7. Graph builder
# ============================================================

def build_data_description_graph(llm: Any):
    graph = StateGraph(DataDescriptionState)

    graph.add_node("load_data", load_data_node)
    graph.add_node("basic_profile", basic_profile_node)
    graph.add_node("llm_schema_review", llm_schema_review_node_factory(llm))
    graph.add_node("quality_checks", quality_checks_node)
    graph.add_node("llm_data_risk_summary", llm_data_risk_summary_node_factory(llm))
    graph.add_node("packaging", packaging_node)

    graph.add_edge(START, "load_data")
    graph.add_edge("load_data", "basic_profile")
    graph.add_edge("basic_profile", "llm_schema_review")
    graph.add_edge("llm_schema_review", "quality_checks")
    graph.add_edge("quality_checks", "llm_data_risk_summary")
    graph.add_edge("llm_data_risk_summary", "packaging")
    graph.add_edge("packaging", END)

    return graph.compile()


# ============================================================
# 8. Public runner
# ============================================================

def run_data_description_agent(
    llm: Any,
    train_path: str,
    test_path: Optional[str] = None,
    sample_submission_path: Optional[str] = None,
    target_column: Optional[str] = None,
    task_type: Optional[str] = None,
    business_task: str = "",
    run_id: Optional[str] = None,
    artifacts_dir: str = "artifacts",
) -> Dict[str, Any]:
    """
    Runs the universal Data Description Agent.

    Args:
        llm:
            LangChain-compatible chat model with .invoke(prompt).
        train_path:
            Path to train CSV.
        test_path:
            Optional path to test CSV.
        sample_submission_path:
            Optional path to sample submission CSV.
        target_column:
            Optional target column. Recommended when train/test inference is ambiguous.
        task_type:
            Optional task type: classification or regression.
        business_task:
            Short business description.
        run_id:
            Optional run ID.
        artifacts_dir:
            Base directory for artifacts.

    Returns:
        Unified response dict compatible with orchestrator.
    """

    if task_type is not None and task_type not in {"classification", "regression"}:
        raise ValueError("task_type must be 'classification', 'regression' or None.")

    run_id = run_id or str(uuid.uuid4())

    initial_state: DataDescriptionState = {
        "run_id": run_id,
        "train_path": train_path,
        "test_path": test_path,
        "sample_submission_path": sample_submission_path,
        "target_column": target_column,
        "task_type": task_type,
        "business_task": business_task,
        "artifacts_dir": artifacts_dir,
        "logs": [
            make_log(
                "data_description_agent",
                f"Universal data description workflow started for run_id={run_id}.",
            )
        ],
        "status": "running",
        "reason": None,
    }

    compiled_graph = build_data_description_graph(llm)
    final_state = compiled_graph.invoke(initial_state)

    summary_path = (
        final_state.get("artifacts", {}) or {}
    ).get("data_description_summary_path")

    if summary_path and Path(summary_path).exists():
        with open(summary_path, "r", encoding="utf-8") as f:
            summary = json.load(f)

        return {
            "agent": "data_description_agent",
            "status": summary.get("status", final_state.get("status")),
            "skipped": False,
            "summary": summary.get("summary", "Data description completed."),
            "decisions": summary.get("decisions", {}),
            "artifacts": summary.get("artifacts", {}),
            "next_input": summary.get("next_input", {}),
            "reason": summary.get("reason"),
        }

    return {
        "agent": "data_description_agent",
        "status": final_state.get("status", "unknown"),
        "skipped": False,
        "summary": "Data description workflow finished, but summary artifact was not found.",
        "decisions": {
            "target_column": final_state.get("target_column"),
            "task_type": final_state.get("task_type"),
            "schema": final_state.get("schema"),
            "data_quality": final_state.get("data_quality"),
        },
        "artifacts": final_state.get("artifacts", {}),
        "next_input": {
            "target_column": final_state.get("target_column"),
            "task_type": final_state.get("task_type"),
            "schema": final_state.get("schema"),
            "data_summary": final_state.get("basic_profile"),
        },
        "reason": final_state.get("reason"),
    }


In [ ]:
result = run_data_description_agent(
    llm=llm,
    train_path="airbnb_train.csv",
    test_path="airbnb_test.csv"
)